In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
import torch, sys
print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("arch list:", torch.cuda.get_arch_list())
print("device:", torch.cuda.get_device_name(0))
print("bf16:", torch.cuda.is_bf16_supported())
print("VRAM:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

import transformers
print("transformers:", transformers.__version__)

In [ ]:
import torch, time

def bench(dtype):
    a = torch.randn(4096, 4096, device="cuda", dtype=dtype)
    b = torch.randn(4096, 4096, device="cuda", dtype=dtype)
    for _ in range(3): a @ b
    torch.cuda.synchronize()
    t = time.time()
    for _ in range(20): a @ b
    torch.cuda.synchronize()
    return (time.time() - t) / 20 * 1000

for dt in [torch.float32, torch.float16, torch.bfloat16]:
    print(f"{dt}: {bench(dt):.1f} ms")

In [ ]:
!git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git /workspace/LIBERO
!pip install -q -e /workspace/LIBERO
!pip install -q robosuite==1.4.1 bddl
!pip install -q "transformers==4.40.1" "tokenizers==0.19.1" "timm==0.9.10" "accelerate>=0.25.0"
!pip install -q numpy==1.26.4

In [ ]:
# ── Cell 3: patch robosuite (3 patch) ──
import robosuite, os, re

# patch 1+2: mj_fullM
p1 = os.path.join(os.path.dirname(robosuite.__file__), "controllers", "base_controller.py")
c = open(p1).read()
c = c.replace(
  'mass_matrix = np.ndarray(shape=(self.sim.model.nv, self.sim.model.nv), dtype=np.float64, order="C")',
  'mass_matrix = np.zeros((self.sim.model.nv, self.sim.model.nv), dtype=np.float64)')
c = re.sub(r"mujoco\.mj_fullM\([^\n]*\)",
           "mujoco.mj_fullM(self.sim.model._model, self.sim.data._data, mass_matrix)", c)
open(p1, "w").write(c)

# patch 3: numpy.int32 vs enum pybind11
p2 = os.path.join(os.path.dirname(robosuite.__file__), "utils", "binding_utils.py")
lines = open(p2).readlines()
n = 0
for i, l in enumerate(lines):
    if "joint_type = self.jnt_type[joint_id]" in l:
        lines[i] = l.replace("self.jnt_type[joint_id]", "int(self.jnt_type[joint_id])")
        n += 1
open(p2, "w").writelines(lines)

!grep -n "mass_matrix = np\|mj_fullM" {p1}
print("jnt_type patched:", n, "(cần = 2)")

In [ ]:
!apt-get update -qq
!apt-get install -y -qq libegl1 libgl1 libglew-dev libosmesa6-dev

In [ ]:
# ── Cell 4: dựng env LIBERO ──
import sys, os
sys.path.insert(0, "/workspace/LIBERO")
os.environ["MUJOCO_GL"] = "egl"

import numpy as np
from libero.libero import benchmark
from libero.libero.envs import OffScreenRenderEnv

task_suite = benchmark.get_benchmark_dict()["libero_spatial"]()
task = task_suite.get_task(0)
print("task:", task.language)

env = OffScreenRenderEnv(
    bddl_file_name=task_suite.get_task_bddl_file_path(0),
    camera_heights=128, camera_widths=128,
)
env.seed(0)
obs = env.reset()
print("image:", obs["agentview_image"].shape)

In [ ]:
!pip install -q easydict thop hydra-core matplotlib opencv-python-headless imageio imageio-ffmpeg

In [ ]:
!pip install -q gym==0.26.2 cloudpickle

In [ ]:
!pip install -q --force-reinstall "scipy==1.13.1"

In [ ]:
!pip install -q hf_transfer

In [ ]:
!df -h /workspace / 
!du -sh /workspace/* 2>/dev/null

In [ ]:
!du -sh /root/.cache/huggingface 2>/dev/null
!du -sh /root/* 2>/dev/null | sort -h | tail -5

In [ ]:
!rm -rf /workspace/.cache
!rm -rf /root/.cache/huggingface

In [ ]:
# ── Cell 5: load OpenVLA ──
import os
os.environ["HF_HOME"] = "/root/hf_cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

from transformers import AutoModelForVision2Seq, AutoProcessor
import torch, time

MODEL_ID = "openvla/openvla-7b-finetuned-libero-spatial"
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

t = time.time()
vla = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
).to("cuda:0").eval()
print(f"loaded {(time.time()-t)/60:.1f} min")

print("VRAM:", torch.cuda.memory_allocated(0)/1e9, "GB")
print("norm stats:", list(vla.norm_stats.keys()))

In [ ]:
# ── Cell 6: một bước predict ──
from PIL import Image
import numpy as np, torch

obs = env.reset()

img = Image.fromarray(obs["agentview_image"][::-1])
prompt = f"In: What action should the robot take to {task.language.lower()}?\nOut:"

inputs = processor(prompt, img).to("cuda:0", dtype=torch.bfloat16)
action = vla.predict_action(**inputs, unnorm_key="libero_spatial", do_sample=False)

print("action:", action)
print("shape:", action.shape, action.dtype)

In [ ]:
# ── Cell 7: rollout (đã sửa thang gripper) ──
from PIL import Image
import numpy as np, torch, time

obs = env.reset()
prompt = f"In: What action should the robot take to {task.language.lower()}?\nOut:"

frames, done = [], False
t = time.time()

for step in range(200):
    img_np = obs["agentview_image"][::-1]
    frames.append(img_np.copy())

    inputs = processor(prompt, Image.fromarray(img_np)).to("cuda:0", dtype=torch.bfloat16)
    action = vla.predict_action(**inputs, unnorm_key="libero_spatial", do_sample=False)

    action[6] = 2 * action[6] - 1
    action[6] = 1 - 2 * action[6]        # ← dòng mới: [0,1] → [-1,1]

    obs, reward, done, info = env.step(action.tolist())
    if done:
        print(f"SUCCESS at step {step}")
        break

print(f"steps: {len(frames)} · done: {done} · {time.time()-t:.0f}s")

In [ ]:
import imageio
imageio.mimsave("/workspace/rollout.mp4", frames, fps=20)
print("saved")

In [ ]:
from IPython.display import Video
Video("/workspace/rollout.mp4", embed=True, width=400)

In [ ]:
grips = []
obs = env.reset()
for step in range(60):
    img = Image.fromarray(obs["agentview_image"][::-1])
    inputs = processor(prompt, img).to("cuda:0", dtype=torch.bfloat16)
    a = vla.predict_action(**inputs, unnorm_key="libero_spatial", do_sample=False)
    grips.append(a[6])
    obs, r, done, info = env.step(a.tolist())

print([round(g, 3) for g in grips])

In [ ]:
for g in [-1.0, 1.0]:
    obs = env.reset()
    for _ in range(30):
        obs, r, d, i = env.step([0,0,0,0,0,0, g])
    print(f"gripper={g:+.1f} → qpos: {obs['robot0_gripper_qpos']}")